# Process Policy Documents to Silver Chunks

This notebook reads TXT and PDF policy files from the UC Volume path, extracts document text, creates overlapping chunks (500 words with 50-word overlap), and writes the result to a Delta silver table.

In [ ]:
%pip install pypdf

In [ ]:
import io
import re
from typing import List

from pypdf import PdfReader
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType

SOURCE_PATH = "dbfs:/Volumes/iitb/bharat_bricks/data/final_submission/policies/"
SOURCE_PATH_ALT = SOURCE_PATH.replace("dbfs:/", "/") if SOURCE_PATH.startswith("dbfs:/") else SOURCE_PATH
TARGET_TABLE = "iitb.bharat_bricks.policy_chunks_silver"
CHUNK_SIZE_WORDS = 500
CHUNK_OVERLAP_WORDS = 50

candidate_paths = [SOURCE_PATH]
if SOURCE_PATH_ALT != SOURCE_PATH:
    candidate_paths.append(SOURCE_PATH_ALT)

loaded_dfs = []
for path_candidate in candidate_paths:
    try:
        candidate_df = (
            spark.read.format("binaryFile")
            .option("recursiveFileLookup", "true")
            .load(path_candidate)
        )
        candidate_count = candidate_df.count()
        print(f"Files discovered at {path_candidate}: {candidate_count}")
        if candidate_count > 0:
            loaded_dfs.append(candidate_df)
    except Exception as exc:
        print(f"Skipping path {path_candidate} because read failed: {exc}")

if not loaded_dfs:
    raise RuntimeError("No files discovered in policy source path using binaryFile.")

raw_files_df = loaded_dfs[0]
for df_part in loaded_dfs[1:]:
    raw_files_df = raw_files_df.unionByName(df_part, allowMissingColumns=True)

raw_files_df = raw_files_df.dropDuplicates(["path"])
print(f"Unique files discovered: {raw_files_df.count()}")

files_df = (
    raw_files_df
    .withColumn("source_name", F.regexp_extract(F.col("path"), r"([^/]+)$", 1))
    .withColumn("file_ext", F.lower(F.regexp_extract(F.col("source_name"), r"\.([^.]+)$", 1)))
    .filter(F.col("file_ext").isin("txt", "pdf"))
)

print(f"Supported txt/pdf files: {files_df.count()}")
display(files_df.select("path", "source_name", "file_ext", "length"))

def extract_document_text(path: str, content: bytes) -> str:
    if content is None:
        return ""

    ext = (path.rsplit(".", 1)[-1] if path and "." in path else "").lower()
    try:
        if ext == "txt":
            return content.decode("utf-8", errors="ignore")

        if ext == "pdf":
            reader = PdfReader(io.BytesIO(content))
            page_text = [(page.extract_text() or "") for page in reader.pages]
            return "\n".join(page_text)
    except Exception:
        return ""

    return ""

extract_text_udf = F.udf(extract_document_text, StringType())

documents_df = (
    files_df
    .withColumn("document_text", extract_text_udf(F.col("path"), F.col("content")))
    .drop("content")
    .filter(F.length(F.trim(F.col("document_text"))) > 0)
)

print(f"Documents with extracted text: {documents_df.count()}")
display(documents_df.select("source_name", "file_ext", F.length("document_text").alias("text_chars")))

In [ ]:
def chunk_words(text: str, chunk_size: int = CHUNK_SIZE_WORDS, overlap: int = CHUNK_OVERLAP_WORDS) -> List[str]:
    words = re.findall(r"\S+", text or "")
    if not words:
        return []

    step = max(chunk_size - overlap, 1)
    chunks = []

    for start in range(0, len(words), step):
        end = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end]).strip()
        if chunk:
            chunks.append(chunk)

        if end >= len(words):
            break

    return chunks

chunk_udf = F.udf(chunk_words, ArrayType(StringType()))

policy_chunks_df = (
    documents_df
    .withColumn("chunks", chunk_udf(F.col("document_text")))
    .select(
        "path",
        "source_name",
        F.posexplode("chunks").alias("chunk_index", "chunk_text")
    )
    .withColumn(
        "chunk_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("source_name"),
                F.col("chunk_index").cast("string"),
                F.col("chunk_text")
            ),
            256
        )
    )
    .withColumn("chunk_word_count", F.size(F.split(F.trim(F.col("chunk_text")), r"\s+")))
    .withColumn("created_at", F.current_timestamp())
    .select(
        "chunk_id",
        "source_name",
        "path",
        "chunk_index",
        "chunk_word_count",
        "chunk_text",
        "created_at"
    )
)

spark.sql("CREATE SCHEMA IF NOT EXISTS iitb.bharat_bricks")

(
    policy_chunks_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Saved table: {TARGET_TABLE}")
print(f"Total chunks: {policy_chunks_df.count()}")
display(policy_chunks_df.orderBy("source_name", "chunk_index").limit(20))